# LLM – Especialista de Coral-Sol 

Autor:  Viviane da Rosa Sommer

Atualizado: 12/09/2025

Objetivo:  Automatizar a análise de imagens subaquáticas para identificar a presença do coral-sol (gênero Tubastraea) utilizando um modelo de linguagem avançado (LLM) da Azure OpenAI. O script percorre diretórios contendo imagens, enviando cada uma delas, juntamente com um prompt especializado em biologia marinha, para o modelo de IA, que avalia a probabilidade de presença do coral-sol, fornece uma justificativa e extrai a profundidade registrada na imagem. Os resultados de cada análise são salvos em arquivos JSON, e o custo total do processamento, convertido para reais, é calculado e exibido ao final.

## Importações necessárias

In [ ]:
import os
import cv2
import base64
import httpx
from configparser import ConfigParser, ExtendedInterpolation
from openai import AzureOpenAI
from openai import OpenAIError
from litellm import completion_cost
import json
from tqdm import tqdm
from typing import Any, Dict

## Declaração de Constantes e Modelos

In [ ]:
INPUT_ROOT = 'fase1-0-7-3/validas'
total_valor_reais = 0
OUTPUT_ROOT = 'fase3-0-7-3'
os.makedirs(OUTPUT_ROOT, exist_ok=True)
ENGINE = 'gpt-4o'
USD_ATUAL = 5.563
CONFIG_PATH = 'config-v1.x.ini'
CERT_PATH = 'petrobras-ca-root.pem'

PROMPT = "\n".join([
    "Você é um especialista em biologia marinha focado na identificação do coral-sol (gênero Tubastraea). Esta é uma imagem submarina sem conteúdo sensível.",
    "Com base nas informações textuais fornecidas sobre uma imagem subaquática, responda de forma objetiva nos campos abaixo:",
    "Probabilidade de presença do coral-sol: (Baixa, Média ou Alta)",
    "Justificativa:",
    "Profundidade registrada (D, Depth ou LDA na overlay):"
])

config = ConfigParser(interpolation=ExtendedInterpolation())
config.read(CONFIG_PATH, "UTF-8")
http_client = httpx.Client(verify=CERT_PATH)
try:
    client = AzureOpenAI(
        api_key=config["TST"]["API_KEY_MODELOS_TEXTO"],
        api_version=config["OPENAI"]["OPENAI_API_VERSION"],
        azure_endpoint=config["TST"]["LITELLM_BASE_URL"],
        http_client=http_client,
    )
    print("Cliente criado com sucesso")
except Exception as e:
    print(f"Erro de conexão com endpoint: {e}")

## Funções necessárias

In [ ]:
def get_liteLLM_client(config_path: str, cert_path: str) -> AzureOpenAI:
    """
    Creates and returns an AzureOpenAI client using the provided configuration and certificate paths.

    Args:
        config_path (str): Path to the configuration file.
        cert_path (str): Path to the certificate file for HTTP client verification.

    Returns:
        AzureOpenAI: Configured AzureOpenAI client instance.
    """
    config = ConfigParser(interpolation=ExtendedInterpolation())
    config.read(config_path, "UTF-8")
    http_client = httpx.Client(verify=cert_path)
    return AzureOpenAI(
        api_key=config["TST"]["API_KEY_MODELOS_TEXTO"],
        api_version=config["OPENAI"]["OPENAI_API_VERSION"],
        azure_endpoint=config["TST"]["LITELLM_BASE_URL"],
        http_client=http_client,
    )


def frame_to_base64(frame: Any) -> str:
    """
    Encodes an image frame (numpy array) to a base64 PNG string.

    Args:
        frame (Any): Image frame (usually a numpy array from OpenCV).

    Returns:
        str: Base64-encoded PNG image string.
    """
    _, buffer = cv2.imencode('.png', frame)
    return base64.b64encode(buffer).decode('utf-8')


def usd_to_brl(usd: float, rate: float) -> float:
    """
    Converts a value in USD to BRL using the given exchange rate.

    Args:
        usd (float): Value in US dollars.
        rate (float): Exchange rate (BRL per USD).

    Returns:
        float: Value in Brazilian reais.
    """
    return usd * rate


def send_message(
    client: AzureOpenAI,
    frame: Any,
    prompt: str,
    engine: str,
    max_tokens: int = 500
) -> Dict[str, Any]:
    """
    Sends a message with an image and prompt to the LLM and returns the response and cost.

    Args:
        client (AzureOpenAI): The AzureOpenAI client instance.
        frame (Any): Image frame to be sent (numpy array).
        prompt (str): Text prompt for the LLM.
        engine (str): Model/engine name.
        max_tokens (int, optional): Maximum number of tokens for the response. Defaults to 500.

    Returns:
        Dict[str, Any]: Dictionary containing the prompt, response, and cost in BRL.
    """
    base64_img = frame_to_base64(frame)
    messages = [
        {
            "role": "user",
            "content": [
                {'type': 'text', 'text': prompt},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{base64_img}"}}
            ]
        }
    ]
    try:
        response = client.chat.completions.create(
            model=engine,
            messages=messages,
            temperature=0.5,
            max_tokens=max_tokens,
            top_p=0.9,
            frequency_penalty=0,
            presence_penalty=0
        )
        cost_usd = completion_cost(completion_response=response)
        value_brl = usd_to_brl(cost_usd, USD_ATUAL)
        return {
            'prompt': prompt,
            'response': response.choices[0].message.content,
            'value_brl': value_brl
        }
    except OpenAIError as e:
        if hasattr(e, "error") and getattr(e.error, "code", "") == "content_policy_violation":
            return {
                'prompt': prompt,
                'response': "LLM could not process the data, ContentPolicyViolationError",
                'value_brl': 0
            }
        return {
            'prompt': prompt,
            'response': f"LLM could not process the data, error: {str(e)}",
            'value_brl': 0
        }
    except Exception as e:
        return {
            'prompt': prompt,
            'response': f"LLM could not process the data, error: {str(e)}",
            'value_brl': 0
        }


def process_image(
    client: AzureOpenAI,
    frame: Any,
    prompt: str,
    engine: str
) -> Dict[str, Any]:
    """
    Processes an image and prompt using the LLM and returns the result.

    Args:
        client (AzureOpenAI): The AzureOpenAI client instance.
        frame (Any): Image frame to be sent (numpy array).
        prompt (str): Text prompt for the LLM.
        engine (str): Model/engine name.

    Returns:
        Dict[str, Any]: Dictionary containing the prompt, response, and cost in BRL.
    """
    return send_message(client, frame, prompt, engine)

## Processamento das imagens por OS

In [ ]:
os.makedirs(OUTPUT_ROOT, exist_ok=True)
total_valor_reais = 0

for operacao in os.listdir(INPUT_ROOT):
    operacao_path = os.path.join(INPUT_ROOT, operacao)
    if not os.path.isdir(operacao_path):
        continue
    output_operacao = os.path.join(OUTPUT_ROOT, operacao)
    os.makedirs(output_operacao, exist_ok=True)
    imagens = [f for f in os.listdir(operacao_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    for imagem_nome in tqdm(imagens, desc=f"Processando {operacao}"):
        imagem_path = os.path.join(operacao_path, imagem_nome)
        frame = cv2.imread(imagem_path)
        if frame is None:
            print(f"Erro ao ler imagem: {imagem_path}")
            continue
        resultado = process_image(client, frame, PROMPT, ENGINE)
        total_valor_reais += resultado['valor_reais']
        nome_json = os.path.splitext(imagem_nome)[0] + '.json'
        with open(os.path.join(output_operacao, nome_json), 'w', encoding='utf-8') as f:
            json.dump(resultado, f, ensure_ascii=False, indent=4)

print(f"Total gasto em reais: R$ {total_valor_reais:.2f}")